# ViT架构源码解析

ViT的核心架构如下图所示：

<div align="center">
    <img src="./imgs/ViT_architecture.png" alt="ViT Architecture" style="width: 750px; height: auto;">
</div>

本篇notebook基于如下链接的内容和代码进行整理，参考链接：

[参考知乎解析](https://zhuanlan.zhihu.com/p/418184940)

[ViT论文](https://arxiv.org/pdf/2010.11929)

[ViT模型Pytorch实现](https://github.com/huggingface/pytorch-image-models/blob/main/timm/models/vision_transformer.py)


## 前置知识的简要介绍


### Self-Attention 与 Multi-Head Attention

#### Self-Attention
给定一个input矩阵(L, D)，通过转换成相同维度的query(Q) key(K) value(V)矩阵(维度与input相同)，然后通过如下公式得到output(L, D)，因为输入输出的维度一样，所以可以进行**堆叠**:

$$
\text{Attention}(Q, K, V) = \text{softmax}\left(\frac{QK^T}{\sqrt{D}}\right) V
$$

其中$\text{softmax}\left(\frac{QK^T}{\sqrt{D}}\right)$得到的是一个注意力图(L, L)，表示不同token间的关系，值越大关系越深;

#### Multi-Head Attention
就是把多个 Self-Attention 头的输出通过concat拼接在一起，拼接后维度为(L, D)，之后再通过输出投影得到最终的输出结果，维度不变:

$$
\text{output} = \text{concat\_output} * W_o  
$$


### Patch Embedding
Patch Embedding 的作用是把图像输入转换为类似(seq_len, dim)这样的形式，从而让 Transoformer 进行作用，在实现中是先进行卷积操作(卷积核大小和步长都等于每个patch的大小 patch size)，展平 H W 两个维度，再把 C 维度和展平的 HW 维度进行交换，就得到了 Patch Embedding，即[B, C, H, W]\(卷积操作后的维度) -> [B, C, HW] -> [B, HW, C]，就类似(seq_len, dim)的形式了

### Transformer Encoder
得到了 Patch Embedding 之后，输入到 Encoder 中，经过 Layer Norm 后输入 Multi-Head Attention 层，再与原始输入相加(在这之前需要经过一个 **Dropout** 层，论文中没有画出)；之后再次经过 Layer Norm 后，输入 MLP Block，经过 **Dropout** 层以后再进行相加操作，就得到了 Encoder 的最终输出，输入输出的维度依旧一致，因此可以堆叠


In [24]:
# 必要库的导入
import torch
import torch.nn as nn
from typing import Any, Callable, Dict, Optional, Set, Tuple, Type, Union, List
from timm.layers import (
    Attention,
    PatchEmbed,
    Mlp,
    LayerNorm,
    LayerType,
    LayerScale,
    DropPath,
    DiffAttention
)
from collections import OrderedDict
from functools import partial


ATTN_LAYERS = {
    '': Attention,
    'attn': Attention,
    'diff': DiffAttention,
}

def _create_attn(
        attn_layer: LayerType,
        dim: int,
        num_heads: int,
        qkv_bias: bool = False,
        qk_norm: bool = False,
        scale_norm: bool = False,
        proj_bias: bool = True,
        attn_drop: float = 0.,
        proj_drop: float = 0.,
        norm_layer: Optional[Type[nn.Module]] = None,
        depth: int = 0,
        **kwargs,
) -> nn.Module:
    if isinstance(attn_layer, str):
        attn_layer = ATTN_LAYERS.get(attn_layer, None)
        assert attn_layer is not None, f'Unknown attn_layer: {attn_layer}'

    # Only pass depth to attention layers that use it
    if issubclass(attn_layer, DiffAttention):
        kwargs['depth'] = depth

    return attn_layer(
        dim,
        num_heads=num_heads,
        qkv_bias=qkv_bias,
        qk_norm=qk_norm,
        scale_norm=scale_norm,
        proj_bias=proj_bias,
        attn_drop=attn_drop,
        proj_drop=proj_drop,
        norm_layer=norm_layer,
        **kwargs,
    )


In [25]:
class Block(nn.Module):
    """Transformer block with pre-normalization."""
    def __init__(
            self,
            dim: int,
            num_heads: int,
            mlp_ratio: float = 4.,
            qkv_bias: bool = False,
            qk_norm: bool = False,
            scale_attn_norm: bool = False,
            scale_mlp_norm: bool = False,
            proj_bias: bool = True,
            proj_drop: float = 0.,
            attn_drop: float = 0.,
            init_values: Optional[float] = None,
            drop_path: float = 0.,  # 意思是说，可能会让残差分支变为全0，即它作用的这一项全为0
            act_layer: Type[nn.Module] = nn.GELU,
            norm_layer: Type[nn.Module] = LayerNorm,
            mlp_layer: Type[nn.Module] = Mlp,
            attn_layer: LayerType = Attention,
            depth: int = 0,
            device=None,
            dtype=None,
    ) -> None:
        """Initialize Block.

            Args:
                dim: Number of input channels.
                num_heads: Number of attention heads.
                mlp_ratio: Ratio of mlp hidden dim to embedding dim.
                qkv_bias: If True, add a learnable bias to query, key, value.
                qk_norm: If True, apply normalization to query and key.
                proj_bias: If True, add bias to output projection.
                proj_drop: Projection dropout rate.
                attn_drop: Attention dropout rate.
                init_values: Initial values for layer scale.
                drop_path: Stochastic depth rate.
                act_layer: Activation layer.
                norm_layer: Normalization layer.
                mlp_layer: MLP layer.
                attn_layer: Attention layer type (class or string).
                depth: Block index, passed to attention layer for depth-dependent init.
            """
        super().__init__()
        dd = {'device': device, 'dtype': dtype}

        self.norm1 = norm_layer(dim, **dd)
        self.attn = _create_attn(
            attn_layer,
            dim,
            num_heads=num_heads,
            qkv_bias=qkv_bias,
            qk_norm=qk_norm,
            scale_norm=scale_attn_norm,
            proj_bias=proj_bias,
            attn_drop=attn_drop,
            proj_drop=proj_drop,  # Dropout概率，在训练时随机丢弃神经元，防止过拟合
            norm_layer=norm_layer,
            depth=depth,
            **dd,
        )
        # LayerScale是可学习的缩放系数，用于对残差分支进行缩放
        self.ls1 = LayerScale(dim, init_values=init_values, **dd) if init_values else nn.Identity()
        self.drop_path1 = DropPath(drop_path) if drop_path > 0. else nn.Identity()

        self.norm2 = norm_layer(dim, **dd)
        # 没有指定out_features，那就和in_features一致
        self.mlp = mlp_layer(
            in_features=dim,
            hidden_features=int(dim * mlp_ratio),
            act_layer=act_layer,
            norm_layer=norm_layer if scale_mlp_norm else None,
            bias=proj_bias,
            drop=proj_drop,
            **dd,
        )
        self.ls2 = LayerScale(dim, init_values=init_values, **dd) if init_values else nn.Identity()
        self.drop_path2 = DropPath(drop_path) if drop_path > 0. else nn.Identity()

    def forward(
            self,
            x: torch.Tensor,
            attn_mask: Optional[torch.Tensor] = None,
            is_causal: bool = False,
    ) -> torch.Tensor:
        x = x + self.drop_path1(self.ls1(self.attn(self.norm1(x), attn_mask=attn_mask, is_causal=is_causal)))
        x = x + self.drop_path2(self.ls2(self.mlp(self.norm2(x))))
        return x
    

## ViT 的实现
**一言以蔽之**，就是把图像变成类似句子的序列，再进行 Transformer 的作用

In [26]:
# 这里直接使用知乎里给出的代码解析，因为源码比较复杂，这里做了很多简化
# 只保留了最核心的代码，以便演示
# NOTE: 想要全面的了解完整复杂的ViT实现，请参考官方的pytorch实现
class VisionTransformer(nn.Module):
    def __init__(self, img_size=224, patch_size=16, in_c=3, num_classes=1000,
                 embed_dim=768, depth=12, num_heads=12, mlp_ratio=4.0, qkv_bias=True,
                 qk_scale=None, representation_size=None, distilled=False, drop_ratio=0.,
                 attn_drop_ratio=0., drop_path_ratio=0., embed_layer=PatchEmbed, norm_layer=None,
                 act_layer=None):
        super(VisionTransformer, self).__init__()

        self.num_classes = num_classes
        self.num_features = self.embed_dim = embed_dim  # num_features for consistency with other models
        self.num_tokens = 2 if distilled else 1  # 一般等于1
        norm_layer = norm_layer or partial(nn.LayerNorm, eps=1e-6)  # 为Norm传入默认参数
        act_layer = act_layer or nn.GELU
 
        self.patch_embed = embed_layer(img_size=img_size, patch_size=patch_size, in_chans=in_c, embed_dim=embed_dim)  # Patch Embedding层
        num_patches = self.patch_embed.num_patches  # patches的总个数
 
        self.cls_token = nn.Parameter(torch.zeros(1, 1, embed_dim))  # 构建可训练参数的0矩阵，用于类别
        self.dist_token = nn.Parameter(torch.zeros(1, 1, embed_dim)) if distilled else None  # 默认为None
        self.pos_embed = nn.Parameter(torch.zeros(1, num_patches + self.num_tokens, embed_dim))  # 位置embedding，和concat后的数据一样
        self.pos_drop = nn.Dropout(p=drop_ratio)  # DropOut层（添加了pos_embed之后）
 
        dpr = [x.item() for x in torch.linspace(0, drop_path_ratio, depth)]  # 从0到ratio，有depth个元素的等差序列
        self.blocks = nn.Sequential(*[
            Block(dim=embed_dim, num_heads=num_heads, mlp_ratio=mlp_ratio, qkv_bias=qkv_bias,
                  proj_drop=drop_ratio, attn_drop=attn_drop_ratio, drop_path=dpr[i],
                  norm_layer=norm_layer, act_layer=act_layer)
            for i in range(depth)  
        ])
        self.norm = norm_layer(embed_dim)
 
        # Representation layer
        if representation_size and not distilled:  # representation_size为True就在MLPHead构建PreLogits,否则只有Linear层
            self.has_logits = True
            self.num_features = representation_size
            self.pre_logits = nn.Sequential(OrderedDict([
                ("fc", nn.Linear(embed_dim, representation_size)),
                ("act", nn.Tanh())
            ]))
        else:
            self.has_logits = False
            self.pre_logits = nn.Identity()
 
        # Classifier head(s) 线性分类层
        self.head = nn.Linear(self.num_features, num_classes) if num_classes > 0 else nn.Identity()
        self.head_dist = None
        if distilled:
            self.head_dist = nn.Linear(self.embed_dim, self.num_classes) if num_classes > 0 else nn.Identity()
 
        # Weight init 权重初始化
        nn.init.trunc_normal_(self.pos_embed, std=0.02)
        if self.dist_token is not None:
            nn.init.trunc_normal_(self.dist_token, std=0.02)
 
        nn.init.trunc_normal_(self.cls_token, std=0.02)
 
    def forward_features(self, x):
        # [B, C, H, W] -> [B, num_patches, embed_dim]
        # [B, 196, 768] 输入Patch Embedding层
        x = self.patch_embed(x)  
        print(f'[forward inner] After patch embedding: {x.size()}')

        # [1, 1, 768] -> [B, 1, 768] 在batch维度复制batch_size份
        cls_token = self.cls_token.expand(x.shape[0], -1, -1)
        if self.dist_token is None:  # 使用蒸馏token的情况，可以先不考虑
            # [B, 197, 768] 将cls_token与x在维度1上拼接，cls_token打头
            x = torch.cat((cls_token, x), dim=1)  
        else:
            x = torch.cat((cls_token, self.dist_token.expand(x.shape[0], -1, -1), x), dim=1)
 
        x = self.pos_drop(x + self.pos_embed)  # concat后的数据加上position，再经过dropout层
        x = self.blocks(x)  # 经过堆叠的Encoder blocks
        x = self.norm(x)  # Layer Norm层

        print(f'[forward inner] After ViT Encoder: {x.size()}')
        
        if self.dist_token is None:  # 一般为None，本质上是Identity层
            # 提取cls_token信息，因为cls_token维度在前，所以索引为0就是cls本身
            return self.pre_logits(x[:, 0])  # 可选的非线性变换，可以理解为降维(e.g. 768->512)
        else:
            return x[:, 0], x[:, 1]
 
    def forward(self, x):
        x = self.forward_features(x)
        
        if self.head_dist is not None:   # 有蒸馏头的情况
            x, x_dist = self.head(x[0]), self.head_dist(x[1])
            if self.training and not torch.jit.is_scripting():
                # during inference, return the average of both classifier predictions
                return x, x_dist
            else:
                return (x + x_dist) / 2
        else:
            x = self.head(x)  # (B, 768) → (B, num_classes)
            print(f'[forward inner] After ViT MLP: {x.size()}')
        return x
    

In [27]:
ViT_model = VisionTransformer(
    img_size=224,           
    patch_size=16,        
    in_c=3,              
    num_classes=1000,  # 类别数
    embed_dim=768,       
    depth=12,           
    num_heads=12,       
    mlp_ratio=4.0,       
    qkv_bias=True,    
    drop_ratio=0.1,        
    attn_drop_ratio=0.1,  # 注意力权重矩阵上应用的Dropout
    drop_path_ratio=0.1, 
)
ViT_model.eval()

x = torch.randn(2, 3, 224, 224)
print(f'Input x size: {x.size()}')
with torch.no_grad():
    output = ViT_model(x)
print(f'Output size: {output.size()}')



Input x size: torch.Size([2, 3, 224, 224])
[forward inner] After patch embedding: torch.Size([2, 196, 768])
[forward inner] After ViT Encoder: torch.Size([2, 197, 768])
[forward inner] After ViT MLP: torch.Size([2, 1000])
Output size: torch.Size([2, 1000])
